# YOLOv8 Controlled Training (v2)

Updated settings:
- epochs = 150
- patience = 40

In [1]:
%pip install -q ultralytics kagglehub pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
import random
import shutil
from pathlib import Path

import kagglehub
import numpy as np
import torch
import yaml
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Local writable workspace (works on macOS/Linux/Colab)
BASE_DIR = (Path.cwd() / "training_artifacts").resolve()
BASE_DIR.mkdir(parents=True, exist_ok=True)

def pick_device():
    if torch.cuda.is_available():
        return 0
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = pick_device()
print(f"Using device: {device}")
print(f"Artifacts dir: {BASE_DIR}")

Using device: mps
Artifacts dir: /Users/matthewgerges/Documents/Waterloo/4A/MSE446/Term Project/boneFracture/training_artifacts


In [3]:
top = kagglehub.dataset_download("pkdarabi/bone-fracture-detection-computer-vision-project", force_download=False)
raw_root = Path(top) / "BoneFractureYolo8"
clean_root = BASE_DIR / "BoneFractureYolo8_clean"

if clean_root.exists():
    shutil.rmtree(clean_root)
shutil.copytree(raw_root, clean_root)

print("Raw:", raw_root)
print("Clean:", clean_root)

Raw: /Users/matthewgerges/.cache/kagglehub/datasets/pkdarabi/bone-fracture-detection-computer-vision-project/versions/2/BoneFractureYolo8
Clean: /Users/matthewgerges/Documents/Waterloo/4A/MSE446/Term Project/boneFracture/training_artifacts/BoneFractureYolo8_clean


In [4]:
def remap_labels(split_dir: Path):
    labels = split_dir / "labels"
    for lf in labels.glob("*.txt"):
        lines = lf.read_text().splitlines()
        out = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            cid = int(parts[0])
            if cid == 4:
                cid = 3
            elif cid == 5:
                cid = 4
            elif cid == 6:
                cid = 5
            parts[0] = str(cid)
            out.append(" ".join(parts))
        lf.write_text("\n".join(out) + ("\n" if out else ""))

for split in ["train", "valid", "test"]:
    remap_labels(clean_root / split)

data_cfg = {
    "path": str(clean_root),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 6,
    "names": [
        "elbow positive",
        "fingers positive",
        "forearm fracture",
        "humerus fracture",
        "shoulder fracture",
        "wrist positive",
    ],
}

yaml_path = BASE_DIR / "data_clean.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)

print(yaml_path)
print(yaml_path.read_text())

/Users/matthewgerges/Documents/Waterloo/4A/MSE446/Term Project/boneFracture/training_artifacts/data_clean.yaml
path: /Users/matthewgerges/Documents/Waterloo/4A/MSE446/Term Project/boneFracture/training_artifacts/BoneFractureYolo8_clean
train: train/images
val: valid/images
test: test/images
nc: 6
names:
- elbow positive
- fingers positive
- forearm fracture
- humerus fracture
- shoulder fracture
- wrist positive



In [5]:
common = dict(
    data=str(yaml_path),
    epochs=150,
    imgsz=640,
    patience=40,
    seed=SEED,
    deterministic=True,
    project=str(BASE_DIR / "runs_controlled"),
    exist_ok=True,
    workers=2,
)

experiments = [
    dict(name="yolov8n_clean", model="yolov8n.pt", batch=16 if device != "cpu" else 8),
    dict(name="yolov8s_clean", model="yolov8s.pt", batch=16 if device != "cpu" else 8),
]

results = []
for exp in experiments:
    print("\nRunning:", exp["name"])
    model = YOLO(exp["model"])
    model.train(
        **common,
        name=exp["name"],
        batch=exp["batch"],
        device=device,
    )
    m = model.val(data=str(yaml_path), split="val", device=device)
    results.append({
        "name": exp["name"],
        "map50": float(m.box.map50),
        "map50_95": float(m.box.map),
        "precision": float(m.box.mp),
        "recall": float(m.box.mr),
    })

print(results)


Running: yolov8n_clean
New https://pypi.org/project/ultralytics/8.4.30 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.13.5 torch-2.10.0 MPS (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/matthewgerges/Documents/Waterloo/4A/MSE446/Term Project/boneFracture/training_artifacts/data_clean.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

KeyboardInterrupt: 

In [ ]:
import json
out = BASE_DIR / "runs_controlled" / "summary.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(results, indent=2))
print("Saved:", out)
print(out.read_text())